In [1]:
import duckdb
import os
import pandas as pd
from dotenv import load_dotenv
from pandas.api.types import is_numeric_dtype

load_dotenv()


file_path   = os.getenv("application_test")
result = duckdb.sql(f"""
    SELECT * 
    FROM read_csv('{file_path}', header=True);
""")

df = pd.DataFrame(pd.read_csv(file_path))


rows_size = 48743

info = df.info()
description = df.describe()


"""

Constant feature Removal

"""

constant_features = [
    col for col in df.columns
    if df[col].nunique(dropna=False) <= 1
]

"""

New DATA frame With constant_features  removed

"""
df = df.drop(constant_features,axis=1)

header = df.columns
header_info = []
for item in header:
    null_sum = df[item].isnull().sum()
    null_sum_ratio = null_sum/rows_size*100
    header_info.append({"name":item,"null sum":null_sum,"null ratio":null_sum_ratio})

"""

DATA QUALITY 

"""
new_df = pd.DataFrame(header_info)
missing_report = new_df.sort_values(
    "null ratio",
    ascending=False
)


high_missing = missing_report[
    (missing_report["null ratio"] > 50)
]

moderate_missing = missing_report[
    (missing_report["null ratio"] >=10) &
    (missing_report["null ratio"] <=50)
]

low_missing = missing_report[missing_report["null ratio"] < 10
]



for column in high_missing['name'] :

    print(f"\n{column}")

    if is_numeric_dtype(df[column]):
        print("Type: Numeric")
        print("Mean:", df[column].mean())
        print("Median:", df[column].median())
        print("Std:", df[column].std())
    else:
        print("Type: Categorical")
        print("Mode:", df[column].mode().tolist())
        print("Unique:", df[column].nunique())



<class 'pandas.DataFrame'>
RangeIndex: 48744 entries, 0 to 48743
Columns: 121 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float64(65), int64(40), str(16)
memory usage: 45.0 MB

COMMONAREA_MEDI
Type: Numeric
Mean: 0.04742038166437143
Median: 0.0223
Std: 0.08289220472380317

COMMONAREA_AVG
Type: Numeric
Mean: 0.047623660567906095
Median: 0.0227
Std: 0.08286838501819505

COMMONAREA_MODE
Type: Numeric
Mean: 0.04522303757623451
Median: 0.0203
Std: 0.08116860365377804

NONLIVINGAPARTMENTS_AVG
Type: Numeric
Mean: 0.009231480158472428
Median: 0.0
Std: 0.048749133634451255

NONLIVINGAPARTMENTS_MODE
Type: Numeric
Mean: 0.008357543677339742
Median: 0.0
Std: 0.04665724760400763

NONLIVINGAPARTMENTS_MEDI
Type: Numeric
Mean: 0.008978853023316233
Median: 0.0
Std: 0.0481484727382851

FONDKAPREMONT_MODE
Type: Categorical
Mode: ['reg oper account']
Unique: 4

LIVINGAPARTMENTS_AVG
Type: Numeric
Mean: 0.10588525432222501
Median: 0.0756
Std: 0.09828404908036381

LIVINGAPARTMENTS_MODE
Type: Nu